# 05 — Field profile validation: 1+1D vs 3+1D

Compares φ(z, t) directly between BubbleMaster (1+1D) and sledgehamr (3+1D)
at three snapshot times per validation point.

**Workflow**
1. Run **Section 1** to list available sledgehamr snapshot times; fill in `SNAP_INDICES`.
2. Run **Section 2** to write BubbleMaster setup files and SLURM scripts.
3. Submit: `! sbatch scripts/fields_<tag>.sh`
4. After jobs complete, run **Sections 3–5** to load and plot.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import _cutoff_times, BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config = pt.Config()
params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

# ---------------------------------------------------------------------------
# Bubble positions (same as notebook 04)
# ---------------------------------------------------------------------------
xlocs84 = [7.225e+1, 1.278e+2, 1.087e+2]
ylocs84 = [8.992e+1, 8.992e+1, 1.101e+2]
zlocs84 = [1.000e+2, 1.000e+2, 1.000e+2]
pos84   = np.column_stack((xlocs84, ylocs84, zlocs84))
dists84 = np.unique(np.linalg.norm(pos84[:, None] - pos84[None], axis=-1))
distance_gamma2 = 1.139e+2 - 8.615e+1

xlocs069 = [1.623e+1, 2.877e+1, 2.408e+1]
ylocs069 = [1.906e+1, 1.906e+1, 2.594e+1]
zlocs069 = [2.250e+1, 2.250e+1, 2.250e+1]
pos069   = np.column_stack((xlocs069, ylocs069, zlocs069))
dists069 = np.unique(np.linalg.norm(pos069[:, None] - pos069[None], axis=-1))

sh_base = '/pscratch/sd/b/buschman/other_runs/'

# ---------------------------------------------------------------------------
# Helper: gamma at a given d for a given model
# ---------------------------------------------------------------------------
def gamma_for(lambda_bar, d):
    model = pt.PhysicsModel(pt.Phi4Potential(lambda_bar), config)
    kin   = model.kinematics
    return float(kin.Gamma(kin.collision_time(d)))

def vp_tag(vp):
    return f'lb{vp["lambda_bar"]}_g{vp["gamma"]:.4g}'

# ---------------------------------------------------------------------------
# Validation points
# sh_N0    : coarsest-level grid points per side
# sh_L     : box side length
# sh_axis  : direction connecting the collision pair ('x'/'y'/'z')
# sh_mid_x : coordinate of the collision midpoint along sh_axis
#             axis shifted so midpoint = 0, matching BM convention where z=0 is midpoint
#             Set to None for runs where the axis/midpoint is not yet determined
# sh_lev   : AMR level (0 = coarsest)
# ---------------------------------------------------------------------------
VALIDATION_POINTS = [
    dict(lambda_bar=0.84, d=distance_gamma2,
         sh_path=sh_base+'bubble_normalization_gamma2',
         sh_N0=2048, sh_L=200., sh_axis='x', sh_mid_x=None, sh_lev=0),
    dict(lambda_bar=0.84, d=dists84[2],
         sh_path=sh_base+'bubble_N3_234_0_2_',
         sh_N0=2048, sh_L=200., sh_axis='x', sh_mid_x=None, sh_lev=0),
    dict(lambda_bar=0.84, d=dists84[3],
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_N0=2048, sh_L=200., sh_axis='x',
         sh_mid_x=(xlocs84[0]+xlocs84[1])/2., sh_lev=0),
    dict(lambda_bar=0.069, d=dists069[1],
         sh_path=sh_base+'bubble_N3_456_1_2_',
         sh_N0=512,  sh_L=50.,  sh_axis='x',
         sh_mid_x=(xlocs069[0]+xlocs069[1])/2., sh_lev=0),
    dict(lambda_bar=0.069, d=dists069[2],
         sh_path=sh_base+'bubble_N3_456_0_2_',
         sh_N0=512,  sh_L=50.,  sh_axis='x', sh_mid_x=None, sh_lev=0),
    dict(lambda_bar=0.069, d=dists069[3],
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_N0=512,  sh_L=50.,  sh_axis='x', sh_mid_x=None, sh_lev=0),
]

# Pre-compute gamma for all VPs so downstream cells don't depend on cell-6 running first
for vp in VALIDATION_POINTS:
    vp['gamma'] = gamma_for(vp['lambda_bar'], vp['d'])
    print(f'lb={vp["lambda_bar"]}  d={vp["d"]:.3f}  gamma={vp["gamma"]:.4g}  tag={vp_tag(vp)}')

# Three snapshot indices to compare — fill in after running Section 1
SNAP_INDICES = [30, 40, 50]

## 1. List sledgehamr snapshot times

In [ ]:
# Run this cell to discover available snapshot times, then update SNAP_INDICES above
for vp in VALIDATION_POINTS:
    output  = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    t_m     = _cutoff_times(vp['d'], params.cutoff_type, params.t_0_scal)[2]
    i_tm    = int(np.argmin(np.abs(t_snaps - t_m)))
    N0_hdr  = int(output._slice_headers[0, 3])
    print(f"lb={vp['lambda_bar']}  d={vp['d']:.3f}  N0={N0_hdr}  {Path(vp['sh_path']).name}")
    for i, t in enumerate(t_snaps):
        marker = ' <-- t_m' if i == i_tm else ''
        print(f"  [{i:3d}]  t = {t:.4f}{marker}")
    print()

## 2. Write BubbleMaster setup and SLURM scripts

One run per validation point with `--save-fields`.  
The full Milne evolution (s from 0 to smax = 1.2·d) is written to `fields.h5`.  
This covers snapshot times up to t_snap ≤ smax (since s = √(t²−z²) ≤ t ≤ smax).

In [ ]:
scripts_dir = REPO_ROOT / 'scripts'
logs_dir    = REPO_ROOT / 'logs'
scripts_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

for vp in VALIDATION_POINTS:
    lb, d = vp['lambda_bar'], vp['d']
    gamma = vp['gamma']
    tag   = vp_tag(vp)

    t_0, t_cut, t_m, _ = _cutoff_times(d, params.cutoff_type, params.t_0_scal)
    smax = 1.2 * d

    model      = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    run_dir    = model.results_dir / 'field_validation' / tag
    setup_path = run_dir / 'setup.h5'
    out_dir    = run_dir / 'out'
    run_dir.mkdir(parents=True, exist_ok=True)

    times = np.array([t_m])
    pt.write_2d_setup(model, gamma=gamma, times=times,
                      path=setup_path, params=params)

    script_path = scripts_dir / f'fields_{tag}.sh'
    script_path.write_text(f"""#!/bin/bash
#SBATCH --constraint=cpu
#SBATCH --nodes=1
#SBATCH --tasks-per-node=1
#SBATCH --cpus-per-task=128
#SBATCH --qos=premium
#SBATCH --time=01:00:00
#SBATCH --job-name=bm_fields_{tag}
#SBATCH --output={logs_dir}/bm_fields_{tag}_%j.out

export OMP_NUM_THREADS=128
export OMP_PLACES=threads
export OMP_PROC_BIND=spread

mkdir -p {out_dir}
/usr/bin/time --verbose srun {config.bubblemaster_bin} {setup_path} {out_dir}/ --save-fields
""")
    print(f'[{tag}]  d={d:.3f}  smax={smax:.3f}  setup→{setup_path}')

print('\nSubmit with:  sbatch scripts/fields_<tag>.sh')

## 3. Helper functions

In [ ]:
def load_bm_fields(out_dir):
    """Return (s, z, phi) from fields.h5; phi has shape (n_s, n_z)."""
    with h5py.File(Path(out_dir) / 'fields.h5', 'r') as f:
        return f['s'][:], f['z'][:], f['phi'][:]


def extract_bm_band(s_bm, z_bm, phi_bm, t_snap):
    """
    For each z, the exact Milne time at Minkowski t_snap is s = sqrt(t_snap² - z²),
    which falls between two BM grid points s[i_lo] and s[i_lo+1].
    Returns (phi_lo, phi_hi) — the field at those two bracketing grid points.
    Points outside the forward light cone (z >= t_snap) are NaN.
    """
    in_cone = z_bm < t_snap
    s_at_t  = np.where(in_cone, np.sqrt(np.maximum(t_snap**2 - z_bm**2, 0.)), 0.)

    i_lo = np.searchsorted(s_bm, s_at_t, side='right') - 1
    i_lo = np.clip(i_lo, 0, len(s_bm) - 2)
    iz   = np.arange(len(z_bm))

    phi_lo = np.where(in_cone, phi_bm[i_lo,     iz], np.nan)
    phi_hi = np.where(in_cone, phi_bm[i_lo + 1, iz], np.nan)
    return phi_lo, phi_hi


def get_sh_field_1d(output, snap_idx, N0, L, mid_x, axis='x', lev=0):
    """
    1D field profile along `axis` at the transverse midpoint row (index N0//2).
    Returns (z_sh, phi_sh, t) with z_sh=0 at the collision midpoint.
    """
    slices = output.GetSlice(snap_idx, axis, lev, ['Phi'])
    t      = slices['t']
    phi_1d = slices['Phi'][N0 * 2**lev // 2, :]
    z_sh   = np.linspace(0, L, N0 * 2**lev) - mid_x
    return z_sh, phi_1d, t

## 4. Load BubbleMaster fields

In [ ]:
bm_data = {}
for vp in VALIDATION_POINTS:
    tag   = vp_tag(vp)
    model = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    out_dir = model.results_dir / 'field_validation' / tag / 'out'
    try:
        s, z, phi = load_bm_fields(out_dir)
        bm_data[tag] = dict(s=s, z=z, phi=phi)
        print(f'[{tag}]  s=[{s[0]:.3f},{s[-1]:.3f}]  '
              f'z=[{z[0]:.3f},{z[-1]:.3f}]  phi shape={phi.shape}')
    except FileNotFoundError:
        print(f'[{tag}]  fields.h5 not found — run BM job first')

## 5. Compare φ(z) at three snapshot times

In [ ]:
n_snaps = len(SNAP_INDICES)
n_vps   = len(VALIDATION_POINTS)
colors  = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

fig, axes = plt.subplots(n_snaps, n_vps,
                          figsize=(4 * n_vps, 3.5 * n_snaps),
                          squeeze=False)

for col, (vp, color) in enumerate(zip(VALIDATION_POINTS, colors)):
    tag   = vp_tag(vp)
    d     = vp['d']
    N0    = vp['sh_N0']
    L     = vp['sh_L']
    mid_x = vp['sh_mid_x']
    bm    = bm_data.get(tag)

    try:
        output = sledgehamr.Output(vp['sh_path'])
        sh_ok  = (mid_x is not None)
    except Exception:
        output = None
        sh_ok  = False

    axes[0, col].set_title(
        rf"$\bar{{\lambda}}$={vp['lambda_bar']}, $\gamma$={vp['gamma']:.3g}",
        fontsize=8)

    for row, snap_idx in enumerate(SNAP_INDICES):
        ax     = axes[row, col]
        t_snap = None

        # --- 3+1D sledgehamr ---
        if sh_ok:
            try:
                z_sh, phi_sh, t_snap = get_sh_field_1d(
                    output, snap_idx, N0, L, mid_x,
                    axis=vp['sh_axis'], lev=vp['sh_lev'])
                mask = z_sh >= 0
                ax.plot(z_sh[mask], phi_sh[mask],
                        color=color, lw=2, label='3+1D')
            except Exception as e:
                ax.text(0.5, 0.5, str(e), ha='center', va='center',
                        transform=ax.transAxes, fontsize=7)
        elif mid_x is None:
            ax.text(0.05, 0.95, 'sh_mid_x not set', transform=ax.transAxes,
                    fontsize=7, va='top', color='gray')

        # --- 1+1D BubbleMaster band ---
        if bm is not None:
            t_band = t_snap if t_snap is not None else _cutoff_times(
                d, params.cutoff_type, params.t_0_scal)[2]
            smax = bm['s'][-1]
            if t_band > smax:
                ax.text(0.05, 0.05,
                        f't={t_band:.2f} > smax={smax:.2f}',
                        transform=ax.transAxes, fontsize=7, color='red', va='bottom')
            else:
                phi_lo, phi_hi = extract_bm_band(bm['s'], bm['z'], bm['phi'], t_band)
                ax.fill_between(bm['z'], phi_lo, phi_hi,
                                color=color, alpha=0.35, linewidth=0, label='1+1D')

        if t_snap is not None:
            ax.text(0.97, 0.95, f't = {t_snap:.3f}',
                    transform=ax.transAxes, ha='right', va='top', fontsize=8)
        ax.set_xlim(0, d * 0.7)
        if col == 0:
            ax.set_ylabel(r'$\phi$')
        if row == n_snaps - 1:
            ax.set_xlabel('z')
        if row == 0 and col == 0:
            ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'validate_fields.pdf', bbox_inches='tight')
plt.show()